# 02 - ECG Signal Preprocessing, Segmentation & AAMI Mapping
## MedAgent: Multimodal Clinical Decision Support System

This notebook demonstrates the end-to-end signal preprocessing pipeline:
1. Zero-phase Butterworth Bandpass & IIR Notch Filtering
2. Heartbeat Window Extraction centered around R-peaks ($260$ samples)
3. AAMI 5-Class Standardization (N, S, V, F, Q)
4. Normalization and Data Augmentations


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import wfdb

from src.config.paths import DATA_RAW_ECG_DIR
from src.data.preprocessing.signal_filtering import preprocess_ecg_signal
from src.data.preprocessing.normalization import zscore_normalize, minmax_normalize, robust_normalize
from src.data.preprocessing.segmentation import extract_heartbeat_segments, AAMI_CLASS_NAMES
from src.data.preprocessing.augmentation import ECGAugmentationPipeline


### 1. Zero-Phase Bandpass & Notch Filtering Demonstration


In [ ]:
record_path = DATA_RAW_ECG_DIR / "mit-bih-arrhythmia-database-1.0.0" / "100"
record = wfdb.rdrecord(str(record_path))
fs = record.fs

# Extract 5 seconds
raw_signal = record.p_signal[:fs * 5, 0]
filtered_signal = preprocess_ecg_signal(raw_signal, fs=fs, lowcut=0.5, highcut=45.0, notch_freq=60.0)

t = np.arange(len(raw_signal)) / fs

fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
axes[0].plot(t, raw_signal, color="#d62728", lw=1.2)
axes[0].set_title("Raw ECG Signal (with Baseline Wander & High-Frequency Noise)", fontsize=13)
axes[0].set_ylabel("Voltage (mV)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, filtered_signal, color="#1f77b4", lw=1.2)
axes[1].set_title("Filtered ECG Signal (0.5-45 Hz Butterworth Zero-Phase + 60 Hz Notch)", fontsize=13)
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Voltage (mV)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 2. Beat Segmentation & AAMI 5-Class Categorization


In [ ]:
# Extract heartbeats across records
annotation = wfdb.rdann(str(record_path), 'atr')
full_filtered = preprocess_ecg_signal(record.p_signal[:, 0], fs=fs)

beats, labels, valid_indices = extract_heartbeat_segments(
    signal=full_filtered,
    r_peaks=annotation.sample,
    annotations=annotation.symbol,
    window_before=90,
    window_after=170
)

print(f"Extracted {len(beats)} heartbeats.")
print(f"Segment shape per beat: {beats[0].shape} samples (at 360 Hz = {260/360*1000:.1f} ms window)")

# Count AAMI classes
unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u} [{AAMI_CLASS_NAMES[u]}]: {c} beats ({c/len(labels)*100:.2f}%)")


### 3. Visualizing Morphology Across AAMI Classes


In [ ]:
plt.figure(figsize=(14, 4))
time_ms = np.linspace(-90/360*1000, 170/360*1000, 260)

# Plot representative beat
plt.plot(time_ms, zscore_normalize(beats[0]), label=f"Normal Beat [N]", color="#1f77b4", lw=2)
plt.axvline(0, color="red", linestyle="--", alpha=0.7, label="R-Peak (t=0)")

plt.title("Extracted Heartbeat Window (260 samples / 722 ms)", fontsize=14)
plt.xlabel("Time relative to R-Peak (ms)", fontsize=12)
plt.ylabel("Normalized Amplitude (Z-Score)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


### 4. ECG Data Augmentations Pipeline


In [ ]:
pipeline = ECGAugmentationPipeline(
    p_noise=1.0,
    p_scale=1.0,
    p_wander=1.0,
    noise_std=0.03,
    scale_range=(0.85, 1.15),
    wander_freq=0.3
)

sample_beat = zscore_normalize(beats[0])
augmented_beat = pipeline(sample_beat, fs=360)

plt.figure(figsize=(14, 4))
plt.plot(time_ms, sample_beat, label="Original Normalized Beat", color="#1f77b4", lw=2)
plt.plot(time_ms, augmented_beat, label="Augmented Beat (Noise + Wander + Scaling)", color="#ff7f0e", linestyle="--", lw=1.5)
plt.title("ECG Augmentations Pipeline Demonstration", fontsize=14)
plt.xlabel("Time relative to R-Peak (ms)")
plt.ylabel("Amplitude")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
